In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn pandas numpy matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.7 MB/s eta 0:00:00


In [ ]:
import torch
import transformers
import datasets
import sklearn
import pandas as pd
import numpy as np

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cpu
Transformers: 5.15.0
Datasets: 4.0.0
GPU available: False


In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("tomaarsen/setfit-absa-semeval-laptops")

print(dataset)

README.md:   0%|          | 0.00/4.79k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  117kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 30.2kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2358 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/654 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'span', 'label', 'ordinal'],
        num_rows: 2358
    })
    test: Dataset({
        features: ['text', 'span', 'label', 'ordinal'],
        num_rows: 654
    })
})


In [ ]:
print(dataset["train"].column_names)

['text', 'span', 'label', 'ordinal']


In [ ]:
print(dataset["train"][0])

{'text': 'I charge it at night and skip taking the cord with me because of the good battery life.', 'span': 'cord', 'label': 'neutral', 'ordinal': 0}


In [ ]:
print("Training samples:", len(dataset["train"]))
print("Test samples:", len(dataset["test"]))

Training samples: 2358
Test samples: 654


In [ ]:
from collections import Counter

sentiments = Counter(dataset["train"]["label"])

print(sentiments)

Counter({'positive': 987, 'negative': 866, 'neutral': 460, 'conflict': 45})


In [ ]:
print(dataset["train"]["span"][:20])

['cord', 'battery life', 'service center', '"sales" team', 'tech guy', 'quality', 'GUI', 'applications', 'use', 'start up', 'features', 'iChat', 'Photobooth', 'garage band', 'features', 'GUI', 'screen', 'power light', 'hard drive light', 'battery']


In [ ]:
from datasets import load_dataset

dataset = load_dataset("tomaarsen/setfit-absa-semeval-laptops")

print(dataset)
print(dataset["train"].column_names)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'span', 'label', 'ordinal'],
        num_rows: 2358
    })
    test: Dataset({
        features: ['text', 'span', 'label', 'ordinal'],
        num_rows: 654
    })
})
['text', 'span', 'label', 'ordinal']
{'text': 'I charge it at night and skip taking the cord with me because of the good battery life.', 'span': 'cord', 'label': 'neutral', 'ordinal': 0}


In [ ]:
print(dataset)
print(dataset["train"].column_names)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'span', 'label', 'ordinal'],
        num_rows: 2358
    })
    test: Dataset({
        features: ['text', 'span', 'label', 'ordinal'],
        num_rows: 654
    })
})
['text', 'span', 'label', 'ordinal']
{'text': 'I charge it at night and skip taking the cord with me because of the good battery life.', 'span': 'cord', 'label': 'neutral', 'ordinal': 0}


In [ ]:
print("Train:", len(dataset["train"]))
print("Test:", len(dataset["test"]))

Train: 2358
Test: 654


In [ ]:
import pandas as pd

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Missing values in training data:")
print(train_df.isnull().sum())

print("\nMissing values in test data:")
print(test_df.isnull().sum())

Missing values in training data:
text       0
span       0
label      0
ordinal    0
dtype: int64

Missing values in test data:
text       0
span       0
label      0
ordinal    0
dtype: int64


In [ ]:
print(train_df["label"].value_counts())

label
positive    987
negative    866
neutral     460
conflict     45
Name: count, dtype: int64


In [ ]:
print(train_df["label"].unique())

['neutral' 'positive' 'negative' 'conflict']


In [ ]:
for i in range(10):
    print(f"\nExample {i+1}")
    print("Review   :", train_df.iloc[i]["text"])
    print("Aspect   :", train_df.iloc[i]["span"])
    print("Sentiment:", train_df.iloc[i]["label"])


Example 1
Review   : I charge it at night and skip taking the cord with me because of the good battery life.
Aspect   : cord
Sentiment: neutral

Example 2
Review   : I charge it at night and skip taking the cord with me because of the good battery life.
Aspect   : battery life
Sentiment: positive

Example 3
Review   : The tech guy then said the service center does not do 1-to-1 exchange and I have to direct my concern to the "sales" team, which is the retail shop which I bought my netbook from.
Aspect   : service center
Sentiment: negative

Example 4
Review   : The tech guy then said the service center does not do 1-to-1 exchange and I have to direct my concern to the "sales" team, which is the retail shop which I bought my netbook from.
Aspect   : "sales" team
Sentiment: negative

Example 5
Review   : The tech guy then said the service center does not do 1-to-1 exchange and I have to direct my concern to the "sales" team, which is the retail shop which I bought my netbook from.
Aspec

In [ ]:
import re

def clean_text(text):
    text = str(text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing spaces
    text = text.strip()

    return text

In [ ]:
example = train_df.iloc[0]["text"]

print("Before:")
print(example)

print("\nAfter:")
print(clean_text(example))

Before:
I charge it at night and skip taking the cord with me because of the good battery life.

After:
I charge it at night and skip taking the cord with me because of the good battery life.


In [ ]:
train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

In [ ]:
train_df[["text", "clean_text", "span", "label"]].head()


,text,clean_text,span,label
0,I charge it at night and skip taking the cord ...,I charge it at night and skip taking the cord ...,cord,neutral
1,I charge it at night and skip taking the cord ...,I charge it at night and skip taking the cord ...,battery life,positive
2,The tech guy then said the service center does...,The tech guy then said the service center does...,service center,negative
3,The tech guy then said the service center does...,The tech guy then said the service center does...,"""sales"" team",negative
4,The tech guy then said the service center does...,The tech guy then said the service center does...,tech guy,neutral


In [ ]:
print(train_df["label"].unique())

['neutral' 'positive' 'negative' 'conflict']


In [ ]:
label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

In [ ]:
train_df["label_id"] = train_df["label"].map(label2id)
test_df["label_id"] = test_df["label"].map(label2id)

In [ ]:
print(train_df[["span", "label", "label_id"]].head(10))

             span     label  label_id
0            cord   neutral       1.0
1    battery life  positive       2.0
2  service center  negative       0.0
3    "sales" team  negative       0.0
4        tech guy   neutral       1.0
5         quality  positive       2.0
6             GUI  positive       2.0
7    applications  positive       2.0
8             use  positive       2.0
9        start up  positive       2.0


In [ ]:
train_df["model_text"] = (
    "Review: " + train_df["clean_text"] +
    " Aspect: " + train_df["span"].astype(str)
)

test_df["model_text"] = (
    "Review: " + test_df["clean_text"] +
    " Aspect: " + test_df["span"].astype(str)
)

In [ ]:
print(train_df["model_text"].iloc[0])

Review: I charge it at night and skip taking the cord with me because of the good battery life. Aspect: cord


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["model_text", "label_id"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["model_text", "label_id"]]
)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['model_text', 'label_id'],
    num_rows: 2358
})
Dataset({
    features: ['model_text', 'label_id'],
    num_rows: 654
})


In [ ]:
print(train_df["label"].unique())
print(train_df["label"].value_counts())
print(train_df[["clean_text", "span", "label"]].head())
print(train_df["model_text"].iloc[0])

['neutral' 'positive' 'negative' 'conflict']
label
positive    987
negative    866
neutral     460
conflict     45
Name: count, dtype: int64
                                          clean_text            span     label
0  I charge it at night and skip taking the cord ...            cord   neutral
1  I charge it at night and skip taking the cord ...    battery life  positive
2  The tech guy then said the service center does...  service center  negative
3  The tech guy then said the service center does...    "sales" team  negative
4  The tech guy then said the service center does...        tech guy   neutral
Review: I charge it at night and skip taking the cord with me because of the good battery life. Aspect: cord


# Transformer-Based Aspect-Based Sentiment Analysis of Product Reviews
## Final Model Development

This section completes the project by:
1. Fine-tuning a Transformer for aspect-level sentiment classification.
2. Evaluating the model using Accuracy, Precision, Recall and F1-score.
3. Plotting a confusion matrix.
4. Testing the model on new reviews.
5. Demonstrating aspect extraction from a raw review using a lightweight noun-phrase/rule-based candidate extractor.

**Important:** The sentiment model uses the annotated aspect spans from SemEval for supervised training. The raw-review aspect extraction demonstration is kept separate so that the project does not claim that the sentiment classifier itself extracts aspects automatically.


In [ ]:
# Optional: verify GPU before training
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Prepare Hugging Face datasets

We use the existing `model_text` created above:

`Review: <review text> Aspect: <aspect>`

The target is the aspect-level sentiment label.


In [ ]:
from datasets import Dataset

# Remove the pandas index column if present
train_dataset = Dataset.from_pandas(
    train_df[["model_text", "label_id"]].reset_index(drop=True)
)
test_dataset = Dataset.from_pandas(
    test_df[["model_text", "label_id"]].reset_index(drop=True)
)

# Trainer expects the target column to be named "labels"
train_dataset = train_dataset.rename_column("label_id", "labels")
test_dataset = test_dataset.rename_column("label_id", "labels")

print(train_dataset)
print(test_dataset)


## 2. Tokenization with BERT

BERT converts the review + aspect pair into tokens and produces contextual representations that are used for sentiment classification.


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["model_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_train = train_dataset.map(tokenize_batch, batched=True)
tokenized_test = test_dataset.map(tokenize_batch, batched=True)

print(tokenized_train[0].keys())


## 3. Load the BERT sentiment classifier

The model has three output classes:

- 0 = Negative
- 1 = Neutral
- 2 = Positive


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print(model.config.id2label)


## 4. Define evaluation metrics

We evaluate the model using Accuracy, Precision, Recall and F1-score.


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


## 5. Fine-tune BERT

The following training configuration is suitable for Google Colab. A GPU is recommended.

If GPU memory is limited, reduce `per_device_train_batch_size` to 8.


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./absa_bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer is ready.")


## 6. Train the model

Run this cell only after all previous cells have executed successfully.


In [ ]:
train_result = trainer.train()

print("Training completed.")
print(train_result)


## 7. Evaluate the trained model

This produces the final Accuracy, Precision, Recall and F1-score on the test set.


In [ ]:
evaluation_results = trainer.evaluate()

print("\nFinal Evaluation Results")
for key, value in evaluation_results.items():
    if isinstance(value, (int, float)):
        print(f"{key}: {value:.4f}")


## 8. Detailed classification report and confusion matrix


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

pred_output = trainer.predict(tokenized_test)
predictions = np.argmax(pred_output.predictions, axis=-1)
true_labels = pred_output.label_ids

print(classification_report(
    true_labels,
    predictions,
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

cm = confusion_matrix(true_labels, predictions)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Negative", "Neutral", "Positive"],
    yticklabels=["Negative", "Neutral", "Positive"]
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("BERT Aspect-Level Sentiment Confusion Matrix")
plt.show()


## 9. Inspect predictions on test reviews

This shows the actual aspect, true sentiment and predicted sentiment.


In [ ]:
results_df = test_df[["clean_text", "span", "label"]].copy()

results_df["predicted_label"] = [
    id2label[int(x)] for x in predictions
]

results_df["correct"] = (
    results_df["label"].str.lower() == results_df["predicted_label"]
)

display(results_df.head(20))
print("Test accuracy from predictions:",
      results_df["correct"].mean())


## 10. Save the trained model

The trained model and tokenizer can be reused later without training again.


In [ ]:
MODEL_OUTPUT_DIR = "./final_absa_bert_model"

trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print("Model saved to:", MODEL_OUTPUT_DIR)


## 11. Create a reusable sentiment prediction function

The model expects both a review and an aspect. This is the supervised ABSA component of the project.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trainer.model.to(device)
trainer.model.eval()

def predict_aspect_sentiment(review, aspect):
    text = f"Review: {clean_text(review)} Aspect: {aspect}"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = trainer.model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred_id = int(torch.argmax(probs).item())

    return {
        "review": review,
        "aspect": aspect,
        "sentiment": id2label[pred_id],
        "confidence": float(probs[pred_id].item())
    }

example_result = predict_aspect_sentiment(
    "The camera quality is excellent, but the battery life is poor.",
    "camera quality"
)

print(example_result)


## 12. Test multiple aspects from one review

This demonstrates the main idea of ABSA: the same review can contain different sentiments for different aspects.


In [ ]:
review = "The camera quality is excellent, but the battery life is poor."

aspects = ["camera quality", "battery life"]

for aspect in aspects:
    result = predict_aspect_sentiment(review, aspect)
    print(f"{aspect} -> {result['sentiment']} "
          f"(confidence: {result['confidence']:.2%})")


## 13. Aspect extraction demonstration for raw reviews

The SemEval dataset provides annotated aspect spans, which were used for supervised sentiment training. To demonstrate the **aspect extraction** stage separately, we use spaCy's noun chunks as candidate aspects.

These candidates are then passed to the trained BERT sentiment classifier.

This is intentionally presented as a **candidate aspect extractor**, not as a claim that BERT automatically extracts aspects in the current classifier.


In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm -q


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_aspect_candidates(review):
    doc = nlp(review)

    candidates = []
    for chunk in doc.noun_chunks:
        phrase = chunk.text.strip()
        if len(phrase.split()) <= 5:
            candidates.append(phrase)

    # Remove duplicates while preserving order
    return list(dict.fromkeys(candidates))

raw_review = "The camera quality is excellent, but the battery life is poor."

candidates = extract_aspect_candidates(raw_review)

print("Candidate aspects:")
for a in candidates:
    print("-", a)


## 14. Complete end-to-end ABSA demonstration

For each extracted candidate aspect, the trained Transformer predicts its sentiment.

Because noun-chunk extraction can produce candidates that are not actual product aspects, the final output should be interpreted as candidate-aspect analysis.


In [ ]:
def analyze_review(review):
    candidates = extract_aspect_candidates(review)
    output = []

    for aspect in candidates:
        result = predict_aspect_sentiment(review, aspect)
        output.append(result)

    return pd.DataFrame(output)

demo_review = "The camera quality is excellent, but the battery life is poor."

display(analyze_review(demo_review))


## 15. Aspect-level sentiment visualization


In [ ]:
demo_df = analyze_review(
    "The camera quality is excellent, but the battery life is poor."
)

plt.figure(figsize=(8, 5))
plt.bar(demo_df["aspect"], demo_df["confidence"])
plt.ylim(0, 1)
plt.ylabel("Prediction Confidence")
plt.xlabel("Aspect")
plt.title("Aspect-Level Sentiment Prediction Confidence")
plt.xticks(rotation=20)
plt.show()


## 16. Final Project Summary

### Input
Product review text.

### Processing
Preprocessing → Aspect information → BERT tokenization → Transformer encoding → Sentiment classification.

### Output
Aspect → Sentiment pairs.

### Example
`Camera Quality → Positive`  
`Battery Life → Negative`

### Evaluation
Accuracy, Precision, Recall, F1-score and Confusion Matrix.

### Technologies
Python, BERT, Hugging Face Transformers, PyTorch, NLTK, Pandas, NumPy, Scikit-learn, Matplotlib, Seaborn, spaCy, Google Colab/VS Code and GitHub.
